# Notebook 06 — Compliance checker evaluation
**Goal:** Measure how accurate `check_compliance()` actually is against real German health-advertising law, using its 4-level graded verdict scale (0=fully compliant, 1=compliant with a minor note, 2=grey area/needs legal review, 3=not compliant), and compare model choices.

By the end of this notebook you will have:
- Run the full compliance system against a hand-labeled test set and an independent holdout set, both on the 4-level scale
- Computed ordinal metrics — exact-match accuracy, off-by-one tolerance, severe-miss rate, mean absolute error — since binary precision/recall doesn't fit a graded scale
- Compared `gpt-4o-mini` against an alternative model on the identical eval sets
- Saved everything to `data/eval/` and regenerated `data/eval/SUMMARY.md`

**Why ordinal metrics, not binary:** predicting verdict 1 when the truth is 0 is a minor miss; predicting 0 when the truth is 3 is a serious one. Binary accuracy treats both as equally wrong. `severe_miss_rate` (|predicted - truth| >= 2) is the number that actually matters for a compliance tool — it's the "called a real violation compliant, or vice versa" rate.

**Verdicts are now Structured Outputs** (a strict JSON schema with an enforced enum), not "JSON mode" — the model cannot return a verdict outside 0-3 or omit a field. See `src/compliance/checker.py`.

**Prerequisite:** Run notebook 05 first so the `eu-regulations` Pinecone namespace is populated — without it, the grounded classifier silently falls back to the ungrounded one and this eval measures the wrong thing.

**Quick reference:** for a fast summary of all past runs without re-running anything, just open `data/eval/SUMMARY.md`.

## Step 1 — Environment check

In [ ]:
import sys
sys.path.append('..')

from src.utils.config import OPENAI_API_KEY, PINECONE_API_KEY, OPENAI_LLM_MODEL

print('✅ OpenAI key loaded:', OPENAI_API_KEY[:8] + '...')
print('✅ Pinecone key loaded:', PINECONE_API_KEY[:8] + '...')
print(f'✅ Default model: {OPENAI_LLM_MODEL}')

## Step 1b — Confirm the legal corpus is loaded

In [ ]:
from pinecone import Pinecone
from src.utils.config import PINECONE_INDEX_NAME
from src.ingestion.legal_docs import LEGAL_NAMESPACE

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)
stats = index.describe_index_stats()
legal_count = stats.get('namespaces', {}).get(LEGAL_NAMESPACE, {}).get('vector_count', 0)

print(f'Legal provisions in \'{LEGAL_NAMESPACE}\': {legal_count}')
assert legal_count > 0, 'Legal corpus is empty — run notebook 05 first.'
print('✅ Legal corpus present, grounded classification will actually be grounded.')

## Step 2 — Load the test sets

In [ ]:
import json
from pathlib import Path
from collections import Counter

main_data = json.loads(Path('../data/eval/compliance_labeled_set.json').read_text(encoding='utf-8'))
holdout_data = json.loads(Path('../data/eval/compliance_holdout_set.json').read_text(encoding='utf-8'))
main_examples = main_data['examples']
holdout_examples = holdout_data['examples']

print(f'Main set: {len(main_examples)} examples — {dict(Counter(e["category"] for e in main_examples))}')
print(f'Holdout set: {len(holdout_examples)} examples — {dict(Counter(e["category"] for e in holdout_examples))}')

## Step 3 — Define which models to evaluate
Runs the identical eval sets through each model listed here. Add or remove models
to compare — e.g. after a prompt change, or to test a newly released model.

In [ ]:
MODELS_TO_EVALUATE = [OPENAI_LLM_MODEL]  # add more model IDs here to compare, e.g. ['gpt-4o-mini', 'gpt-5.6-terra']
print(f'Evaluating: {MODELS_TO_EVALUATE}')

## Step 4 — Run the eval for each model, on both sets

In [ ]:
from src.utils.compliance_eval import run_compliance_eval, compute_ordinal_metrics, breakdown_by_category

all_runs = []  # one entry per (model, set_type)

for model in MODELS_TO_EVALUATE:
    for set_type, examples in [('main', main_examples), ('holdout', holdout_examples)]:
        print(f'=== {model} — {set_type} set ({len(examples)} examples) ===')
        results = run_compliance_eval(examples, model=model)
        for i, r in enumerate(results, 1):
            mark = '✓' if r['correct'] else '✗'
            print(f'  [{i}/{len(results)}] {mark} {r["id"]} truth={r["ground_truth_verdict"]} pred={r["predicted_verdict"]} src={r["source"]}')
        metrics = compute_ordinal_metrics(results)
        print(f'  Metrics: {metrics}\n')
        all_runs.append({
            'model': model, 'set_type': set_type,
            'examples': examples, 'results': results,
            'metrics': metrics, 'by_category': breakdown_by_category(results),
        })

print('✅ All eval runs complete.')

## Step 5 — Compare results across models

In [ ]:
print(f'{"Model":<18}{"Set":<10}{"Exact":>8}{"Off-by-1":>10}{"Severe":>9}{"MAE":>8}')
for run in all_runs:
    m = run['metrics']
    print(f'{run["model"]:<18}{run["set_type"]:<10}{m["exact_match_accuracy"]:>8.1%}{m["off_by_one_accuracy"]:>10.1%}{m["severe_miss_rate"]:>9.1%}{m["mean_absolute_error"]:>8.3f}'
    )

## Step 6 — Save results (archive + latest) and regenerate the summary

In [ ]:
from datetime import datetime, timezone
from src.utils.eval_summary import generate_eval_summary

runs_dir = Path('../data/eval/runs')
runs_dir.mkdir(parents=True, exist_ok=True)

for run in all_runs:
    output = {
        'run_at': datetime.now(timezone.utc).isoformat(),
        'model': run['model'],
        'set_type': run['set_type'],
        'note': f"4-level verdict scale, Structured Outputs + few-shot exemplars, model={run['model']}.",
        'metrics': run['metrics'],
        'by_category': run['by_category'],
        'predictions': run['results'],
    }

    latest_name = 'compliance_holdout_results.json' if run['set_type'] == 'holdout' else 'compliance_results.json'
    Path(f'../data/eval/{latest_name}').write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding='utf-8')

    label = output['run_at'].replace(':', '').replace('.', '')[:15]
    model_tag = run['model'].replace('.', '').replace('-', '')
    archive_path = runs_dir / f"{label}_{model_tag}_{run['set_type']}.json"
    archive_path.write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f'✅ Saved {archive_path.name}')

summary_path = generate_eval_summary()
print(f'\n✅ Regenerated {summary_path}')
print(f'\n{summary_path.read_text(encoding="utf-8")}')

## Notes

**What this does and doesn't prove:** this measures graded verdict accuracy against
hand-labeled sets — it does not validate that cited section numbers are legally
precise. Treat the numbers as directional evidence, not a legal certification.

**Model comparisons and prompt bias:** if you add a new model to `MODELS_TO_EVALUATE`,
keep in mind the classifier prompt and few-shot examples were iteratively tuned against
`gpt-4o-mini`'s behavior — a new model may simply need its own prompt calibration to be
judged fairly, so a lower score on first try isn't necessarily a worse model, just an
untuned one. State this caveat when reporting a comparison.

**Re-running:** Safe any time. `compliance_results.json` / `compliance_holdout_results.json`
always reflect the latest run; every run is archived under
`data/eval/runs/<timestamp>_<model>_<set>.json` and indexed in `data/eval/SUMMARY.md`.

**Extending either test set:** Add examples to `compliance_labeled_set.json` (main/fitted
set) or `compliance_holdout_set.json` (generalization check), with a `ground_truth_verdict`
(0-3), `category`, and `rationale`. Keep them non-overlapping — the holdout set only means
something if its examples were never looked at while designing a fix.